# **工程化算子开发介绍**

本节为Ascend C算子工程化开发介绍，将依据算子工程化实现流程，对各步骤要点进行讲解。重点学习Host侧的完整实现，包括Tiling实现、Shape推导等函数实现，以及算子原型注册。Kernel侧实现将主要讲解其在工程化框架下的适配要点，与之前独立核函数编程的差异。我们将按照以下结构，带你学习算子工程开发：
- 环境准备
- 什么是算子工程
- 工程化算子实现流程
- 如何创建算子工程原型定义文件
- 如何创建算子工程
- 算子Host侧实现
- 算子Kernel侧实现
- 课后实践 

---

## **1. 环境准备**

本文所有内容均存放于Sources文件夹。
在开始创建算子工程前，先要对jupyter环境进行初始化。以下代码完成了初始化并将环境中的变量导入jupyter环境，并完成代码目录的创建。保证能正常导入代码以及msopgen工具。


In [ ]:
!mkdir -p Sources/03.02

import os, subprocess
env = subprocess.check_output("bash -l -c 'source $ASCEND_TOOLKIT_HOME/set_env.sh && env'", shell=True, text=True)
for line in env.splitlines():
    if "=" in line: os.environ.__setitem__(*line.split("=", 1))
print("\n🎉 Environment initialization process completed successfully!")

---
## **2. 什么是算子工程**
与此前仅聚焦kernel侧核函数实现的开发模式不同，Ascend C单算子工程是一种标准化的开发范式。它实现了从Host侧的算子注册、形状推导、Tiling分块、任务下发与内存管理，到kernel侧使用 Ascend C编写核函数计算逻辑的完整流程，最终生成一个可直接编译和部署的完整算子包。
CANN开发套件包中提供了自定义算子工程生成工具msOpGen，可基于算子原型定义输出算子工程：包括算子host侧代码实现文件、算子kernel侧实现文件以及工程编译配置文件等。

---

## **3. 工程化算子实现流程**
工程化算子开发包含CANN环境部署、算子工程创建、算子实现、编译部署、调用测试。其中算子实现内的kernel侧算子实现与之前课程的算子实现无太大差异，本节课不做详细介绍，重点介绍Host侧算子实现。  </br>
<img src="./images/operator_implementation_flow.png"  alt="operator_implementation_flow" />

---


## **4. 如何创建算子工程原型定义文件**
本节我们开发自定义算子AddCustomTemplate：该算子功能等价于原生Add算子，输入输出张量为ND格式，支持float16、float32数据类型。首先，我们将算子类型定义为AddCustomTemplate。

In [ ]:
%%writefile Sources/03.02/add_custom.json
[
    {
        "op": "AddCustomTemplate",


接下来，我们需要明确该算子的输入张量规范，包括张量格式与数据类型：原生Add算子包含两个输入张量，对应到我们的自定义算子中，我们将这两个输入分别命名为x和y；同时要求这两个输入均需支持两种数据规格,ND格式的float16类型、以及ND格式的float32类型。

In [ ]:
%%writefile -a Sources/03.02/add_custom.json
        "input_desc": [
            {
                "name": "x",
                "param_type": "required",
                "format": ["ND", "ND"],
                "type": ["float16", "float"]
            },
            {
                "name": "y",
                "param_type": "required",
                "format": ["ND", "ND"],
                "type": ["float16", "float"]
            }
        ],

完成输入张量的定义后，我们继续明确输出张量的规范：Add算子仅有一个输出张量，我们将其命名为z；该输出张量同样需满足格式与类型，支持ND格式的float16类型、以及ND格式的float32类型。

In [ ]:
%%writefile -a Sources/03.02/add_custom.json
        "output_desc": [
            {
                "name": "z",
                "param_type": "required",
                "format": ["ND", "ND"],
                "type": ["float16", "float"]
            }
        ]
    }
]

---

## **5. 如何创建算子工程**
基于上文已定义完成的算子原型定义文件，我们可借助msOpGen工具快速创建名为custom_op的自定义算子工程。

### **5.1 准备算子工程创建工具**
执行以下命令安装msOpGen工具：

In [ ]:
# 下载工具仓代码
!git clone https://gitcode.com/Ascend/msopgen.git
#编译安装msopgen工具
!cd msopgen; python build.py;cd output;pip install mindstudio_opgen*.whl --force-reinstall;pip install mindstudio_opst*.whl --force-reinstall


### **5.2 创建算子工程**
准备好msopgen工具后，执行以下命令创建算子工程：

In [ ]:
# 清除已有的custom_op目录
!rm -rf Sources/03.02/custom_op

# 使用msopgen工具创建算子工程前设置使用开源仓编译的msopgen工具，不设置时默认使用CANN安装目录下的msopgen工具
!export PATH=/usr/local/bin:~/.local/bin:$PATH;msopgen gen -i Sources/03.02/add_custom.json -c ai_core-ascend910b1 -lan cpp -out Sources/03.02/custom_op


命令参数含义：  
- -i：指定算子原型定义文件add_custom.json所在路径，请根据实际情况修改。
- -c：指定算子在对应的昇腾AI处理器上执行。
- -lan：参数cpp代表算子基于Ascend C编程框架，使用C/C++编程语言开发。
- -out：生成算子工程所在路径，可配置为绝对路径或者相对路径。

### 算子工程目录结构
命令执行完后，会在-out指定目录或者默认路径下生成算子工程目录，工程中包含算子实现的模板文件，编译脚本等，以AddCustomTemplate算子为例，目录结构如下所示：
```
custom_op
|--- framework
|--- op_host
|   |--- add_custom_template.cpp          // 包含算子原型注册、tiling实现 shape与Dtype推导内容
|   |--- CMakeLists.txt                   // host侧CMakeLists文件，一般不需要修改
|--- op_kernel
|   |--- add_custom_template_tiling.h     // 算子tiling定义文件   
|   |--- add_custom_template.cpp          // 算子代码实现文件 
|   |--- CMakeLists.txt                   // kernel侧CMakeLists文件，一般不需要修改
|--- CMakeLists.txt                       // 根目录CMakeLists文件，一般不需要修改
|--- CMakePresets.json                    // CMake编译配置文件
|--- build.sh                             // 编译脚本
```  
工程目录中的op_kernel和op_host包含了算子的核心实现文件。op_kernel下存放kernel侧算子实现。op_host下存放host侧代码实现，包括算子原型定义、host侧tiling实现。
* custom_op: msopgen工具创建算子工程时指定的算子工程名，可自定义
* **op_host/add_custom_template.cpp**: 文件名会根据add_custom.json内定义的算子名生成，包含算子原型注册、tiling实现 Shape与Dtype推导内容。
* **op_kernel/add_custom_template_tiling.h**: 文件名会根据add_custom.json内定义的算子名生成，包含算子tiling结构体定义。
* **op_kernel/add_custom_template.cpp**: 文件名会根据add_custom.json内定义的算子名生成，包含算子代码实现。
* **CMakePresets.json**: CMake编译配置文件，一般只需要修改ASCEND_CANN_PACKAGE_PATH为实际CANN安装路径。

执行以下命令查看刚刚生成的算子工程目录结构：

In [ ]:
!cd Sources/03.02/custom_op;find . -maxdepth 2 -print | sed -e 's;[^/]*/;|____;g;s;____|;    |;g'

---

## **6 算子Host侧实现**
从msopgen工具生成的算子工程中，我们可以看到op_host和op_kernel目录，分别存放了算子的host侧和kernel侧实现代码。
在Ascend C算子工程中，Host侧是算子执行的控制与管理层，其核心作用是弥补kernel侧仅擅长高密度并行计算的能力短板，具体承担三类关键工作：
* 算子原型注册: 注册算子的原型定义，从而确保算子能够被框架正确识别、编译和执行。算子原型主要描述了算子的输入输出、属性等信息以及算子在AI处理器上相关实现信息，并关联tiling实现等函数。  

* Shape、Dtype推导函数实现: 算子参数的校验，实现程序健壮性并提高定位效率。根据算子的输入张量描述、算子逻辑及算子属性，推理出算子的输出张量描述，包括张量的形状、数据类型及数据排布格式等信息。这样算子构图准备阶段就可以为所有的张量静态分配内存，避免动态内存分配带来的开销。  

* Tiling实现: 大多数情况下，Local Memory的存储，无法完整的容纳算子的输入与输出，需要每次搬运一部分输入进行计算然后搬出，再搬运下一部分输入进行计算，直到得到完整的最终结果，这个数据切分、分块计算的过程称之为Tiling。根据算子的shape等信息来确定数据切分算法相关参数（比如每次搬运的块大小，以及总共循环多少次）的计算程序，称之为Tiling实现

执行以下命令查看算子工程创建时默认生成的算子host侧代码实现：

In [ ]:
!cat Sources/03.02/custom_op/op_host/add_custom_template.cpp


### 6.1 算子原型注册
在上文生成的AddCustomTemplate算子工程目录中，算子原型注册代码在add_custom_template.cpp文件中，创建算子工程时会自动生成，通常不需要修改。  

* 算子原型注册代码内用OP_ADD(AddCustomTemplate)注册了AddCustomTemplate算子，该算子支持ND格式的float16输入数据运算，  并输出ND格式的float16数据，支持ND格式的float32输入数据运算，并输出ND格式的float32数据。  

* 算子类通过SetInferShape指定了AddCustomTemplate算子的Shape推导函数为InferShape。  

* 算子类通过SetInferDataType指定了AddCustomTemplate算子的DataType推导函数为InferDataType。  

* 算子类通过SetTiling指定了AddCustomTemplate算子的Tiling实现函数为TilingFunc。  

* 算子类通过AddConfig指定了支持AddCustomTemplate算子的昇腾设备型号。  

```

namespace ops {
class AddCustomTemplate : public OpDef {
public:
    explicit AddCustomTemplate(const char *name) : OpDef(name)
    {
        this->Input("x")
            .ParamType(REQUIRED)
            .DataType({ge::DT_FLOAT16, ge::DT_FLOAT})
            .Format({ge::FORMAT_ND, ge::FORMAT_ND});
        this->Input("y")
            .ParamType(REQUIRED)
            .DataType({ge::DT_FLOAT16, ge::DT_FLOAT})
            .Format({ge::FORMAT_ND, ge::FORMAT_ND});
        this->Output("z")
            .ParamType(REQUIRED)
            .DataType({ge::DT_FLOAT16, ge::DT_FLOAT})
            .Format({ge::FORMAT_ND, ge::FORMAT_ND});

        this->SetInferShape(ge::InferShape).SetInferDataType(ge::InferDataType);
        this->AICore()
            .SetTiling(optiling::TilingFunc)
            .AddConfig("ascend910b");
    }
};
OP_ADD(AddCustomTemplate);
} 
```

### 6.2 算子Shape推导函数实现
Shape 推导、Dtype 推导等函数会依据算子的输入张量描述、算子核心逻辑及算子属性，精准推理出算子的输出张量描述，涵盖张量的 Shape、数据类型、数据排布格式等关键信息。基于该推导结果，算子构图准备阶段可提前为所有张量完成静态内存分配，从根源上规避动态内存分配带来的性能开销。其整体执行流程可简化为：  

1. 确定全图首节点的TensorDesc，按节点连接关系向下逐个传播；

2. GE将前一算子的输出Tensor信息，同步刷新至下一算子的输入Tensor；  

3. 算子调用InferShape等推导函数，根据输入TensorDesc完成输出TensorDesc的推导与更新；  

4. 按上述逻辑逐节点依次推导，直至完成全图所有算子的张量描述推导。 

不考虑广播的情况下，AddCustomTemplate算子的输入输出应该shape一致，所以shape推导函数内可以设置输出shape等于输入shape。  
```

static graphStatus InferShape(gert::InferShapeContext *context)
{
    const gert::Shape *intputShape = context->GetInputShape(0);
    gert::Shape *outputShape = context->GetOutputShape(0);
    *outputShape = *intputShape;
    return GRAPH_SUCCESS;
}
```

### 6.3 算子Dtype推导函数实现
Dtype推导等函数实现和Shape推导函数会根据算子的输入张量描述、算子逻辑及算子属性，推理出算子的输出张量描述，包括张量的数据类型等信息。这样算子构图准备阶段就可以为所有的张量静态分配内存，避免动态内存分配带来的开销。不考虑不同类型数据相加的情况下，AddCustomTemplate算子的输入输出数据类型一致，所以Dtype推导函数内可以设置输出Dtype等于输入Dtype。  
```

static graphStatus InferDataType(gert::InferDataTypeContext *context)
{
    context->SetOutputDataType(0, context->GetInputDataType(0));
    return ge::GRAPH_SUCCESS;
}
```

## 6.4 算子Tiling函数实现
上文提及了算子原型注册是通过setTiling指定了算子的tiling函数，那么什么是tiling函数呢？带着这个问题，我们来详细了解一下tiling函数。

要实现tiling函数，首先我们需要知道什么是tiling。
在NPU架构中，AI Core的片上存储容量有限，如果输入数据量较大时，会存在无法一次性加载算子全部输入、输出与中间计算数据的情况。因此需要将完整数据按计算粒度切分为多个小块，通过 “搬运→计算→回写” 的循环流水线完成全量计算，这个数据分块、分段计算的过程，就是 Tiling。</br>
简单来说：受限于片上存储大小，把 “一次装不下、算不完” 的大数据，拆成多块小数据分批处理，就是 Tiling。</br>

接下来，我们通过一个直观的示例来理解tiling机制的核心作用：  

- 设定场景：假设某 AI Core 的片上存储容量上限为 10 个数据元素，我们需要完成一个长度为 10 的向量加法运算（输入张量 A + 输入张量 B = 输出张量 C）。  

- 核心约束：运算过程中，输入张量（A、B）与输出张量（C）需同时驻留在 AI Core 的片上存储中，因此每次计算时，三者的总元素个数必须≤片上存储容量（10 个元素）。  

- 分片计算逻辑：基于上述约束，单次可处理的最大数据块大小为 `max_once_num = 10 // 3 = 3`（3 个元素对应A、B、C各3个，总占用9个存储位置，符合容量限制）；整个向量加法需切分为 `tile_num = (10 + 3 - 1) // 3 = 4` 次完成（向上取整），最终运算结果与整段数据直接计算完全一致。 

整体执行流程如图：  
<img src="./images/tile_animation_slow.gif" alt="tiling" width="700px" />


In [ ]:
import numpy as np

def tiling_array_add(arr1, arr2, tile_num, arr_length):
    result = np.zeros_like(arr1)
    tile_size = arr_length // tile_num
    print("开始Tiling分块相加...")
    for start_idx in range(0, arr_length, tile_size):
        end_idx = min(start_idx + tile_size, arr_length)
        tile1 = arr1[start_idx:end_idx]
        tile2 = arr2[start_idx:end_idx]
        tile_result = tile1 + tile2
        result[start_idx:end_idx] = tile_result
        print(f"  完成Tile：arr1[{start_idx}:{end_idx}] + arr2[{start_idx}:{end_idx}] = {tile_result}")
    print("Tiling分块相加完成！")
    return result

if __name__ == "__main__":
    max_ub = 10
    arr_length = 10
    input_output_num = 3
    max_once_num = max_ub // input_output_num
    tile_num = arr_length // max_once_num  # 直接指定分块数量
    arr_a = np.arange(1, arr_length+1, dtype=np.int32)
    arr_b = np.full(arr_length, 5, dtype=np.int32)
    print("=" * 40)
    print("原始数组arr_a：", arr_a)
    print("原始数组arr_b：", arr_b)
    print("=" * 40)
    result_direct = arr_a + arr_b
    print("直接相加结果：", result_direct)
    print("=" * 40)
    result_tiling = tiling_array_add(arr_a, arr_b, tile_num, arr_length)
    print("=" * 40)
    print("Tiling分块结果：", result_tiling)
    print("=" * 40)

Tiling实现完成后，其切分算法生成的相关参数，会被传递至Kernel侧，用于指导并行数据的切分与调度。在上述示例代码中，我们可将tiling_array_add视为核函数，而该核函数在执行运算时所需的单次运算数据量tiling_size，正是在Host侧通过Tiling函数完成计算后，再传入核函数的Tiling相关数据。

那么我们如何像上文示例那样，告诉核函数输入的长度以及切分次数呢？首先我们需要定义传入核函数的Tiling结构体, 用于存储Tiling相关数据。假设我们需要让核函数知道输入向量的长度，那么我就需要在Tiling结构体中添加一个成员变量totalLength，用于存储输入向量的长度。另外我们也需要在Tiling结构体中添加一个成员变量tileNum，用于存储切分次数。  

In [ ]:
%%writefile  Sources/03.02/custom_op/op_kernel/add_custom_template_tiling.h
#ifndef ADD_CUSTOM_TEMPLATE_TILING_H
#define ADD_CUSTOM_TEMPLATE_TILING_H
#include <cstdint>

struct TilingDataTemplate {
    uint32_t totalLength;
    uint32_t tileNum;
};
#endif

接着我们需要在Host侧实现Tiling函数，获取输入向量的长度，并将其存储到Tiling结构体中，同时通过计算获取或者直接指定切分次数，将切分次数存储到Tiling结构体中。  </br>

```

static ge::graphStatus TilingFunc(gert::TilingContext *context)
{
    uint32_t totalLength = context->GetInputShape(0)->GetOriginShape().GetShapeSize();
    context->SetBlockDim(8);
    TilingDataTemplate *tiling = context->GetTilingData<TilingDataTemplate>();
    tiling->totalLength = totalLength;
    tiling->tileNum = 1;
    return ge::GRAPH_SUCCESS;
}
```
这样我们就完成了一个简单的Tiling函数实现，这个函数会获取算子第一个输入的元素个数，并将其存储到Tiling结构体的totalLength字段中，同时也将切分次数存储到Tiling结构体的tileNum字段中。

完成TIling实现后，整体host侧代码如下：

In [ ]:
%%writefile Sources/03.02/custom_op/op_host/add_custom_template.cpp 
#include "register/op_def_registry.h"
#include "../op_kernel/add_custom_template_tiling.h"

namespace optiling {

static ge::graphStatus TilingFunc(gert::TilingContext *context)
{
    uint32_t totalLength = context->GetInputShape(0)->GetOriginShape().GetShapeSize();
    context->SetBlockDim(8);
    TilingDataTemplate *tiling = context->GetTilingData<TilingDataTemplate>();
    tiling->totalLength = totalLength;
    tiling->tileNum = 1;
    return ge::GRAPH_SUCCESS;
}
}  // namespace optiling

namespace ge {
static graphStatus InferShape(gert::InferShapeContext *context)
{
    const gert::Shape *intputShape = context->GetInputShape(0);
    gert::Shape *outputShape = context->GetOutputShape(0);
    *outputShape = *intputShape;
    return GRAPH_SUCCESS;
}

static graphStatus InferDataType(gert::InferDataTypeContext *context)
{
    context->SetOutputDataType(0, context->GetInputDataType(0));
    return ge::GRAPH_SUCCESS;
}
}  // namespace ge

namespace ops {
class AddCustomTemplate : public OpDef {
public:
    explicit AddCustomTemplate(const char *name) : OpDef(name)
    {
        this->Input("x")
            .ParamType(REQUIRED)
            .DataType({ge::DT_FLOAT16, ge::DT_FLOAT})
            .Format({ge::FORMAT_ND, ge::FORMAT_ND});
        this->Input("y")
            .ParamType(REQUIRED)
            .DataType({ge::DT_FLOAT16, ge::DT_FLOAT})
            .Format({ge::FORMAT_ND, ge::FORMAT_ND});
        this->Output("z")
            .ParamType(REQUIRED)
            .DataType({ge::DT_FLOAT16, ge::DT_FLOAT})
            .Format({ge::FORMAT_ND, ge::FORMAT_ND});

        this->SetInferShape(ge::InferShape).SetInferDataType(ge::InferDataType);
        this->AICore()
            .SetTiling(optiling::TilingFunc)
            .AddConfig("ascend910b");
    }
};
OP_ADD(AddCustomTemplate);
} 

---

## **7 算子Kernel侧实现**
与之前核函数的课程中介绍的核函数实现稍微有一些不同，算子工程的核函数会多出workspace和tiling参数，其中tiling参数即为上文Tiling函数中存储的Tiling结构体数据。所以在核函数中需要先通过REGISTER_TILING_DEFAULT和GET_TILING_DATA_WITH_STRUCT宏来获取Tiling结构体数据，然后根据Tiling结构体数据来实现算子的计算。例如我们之前在Tiling结构体内定义了totalLength和tileNum，这里我们就可以通过tiling_data.totalLength和tiling_data.tileNum来获取tiling函数实现内设置的值。
- REGISTER_TILING_DEFAULT：注册TilingData结构体用于告知框架侧用户使用标准C++语法来定义TilingData，同时告知框架TilingData结构体类型，用于框架做tiling数据解析。  

- GET_TILING_DATA_WITH_STRUCT：获取指定的tiling信息，并填入对应的Tiling结构体中，第一个参数为TilingData的结构体类型，第二个参数为指定Tiling结构体变量，第三个参数为核函数处传入的tiling参数。

```

__global__ __aicore__ void add_custom_template(GM_ADDR x, GM_ADDR y, GM_ADDR z, GM_ADDR workspace, GM_ADDR tiling)
{
    REGISTER_TILING_DEFAULT(TilingDataTemplate);
    GET_TILING_DATA_WITH_STRUCT(TilingDataTemplate, tiling_data, tiling);
    KernelAdd<DTYPE_X, DTYPE_Y, DTYPE_Z> op;
    op.Init(x, y, z, tiling_data.totalLength, tiling_data.tileNum);
    op.Process();
}
```


算子工程的算子类实现与之前核函数的课程中介绍的算子类实现类似，不同的是算子工程支撑动态shape输入，所以在Init函数中我们需要根据Tiling结构体数据来动态初始化内存，而不是使用固定的变量进行初始化。

- 核函数实现需要先设置好Tiling相关参数，再进行初始化操作
```
    constexpr int32_t TOTAL_LENGTH = 8 * 2048;                           
    constexpr int32_t USE_CORE_NUM = 8;                                  
    constexpr int32_t BLOCK_LENGTH = TOTAL_LENGTH / USE_CORE_NUM;        
    constexpr int32_t TILE_NUM = 8;                                      
    constexpr int32_t BUFFER_NUM = 2;                                    
    constexpr int32_t TILE_LENGTH = BLOCK_LENGTH / TILE_NUM / BUFFER_NUM;
    __aicore__ inline void Init(GM_ADDR x, GM_ADDR y, GM_ADDR z)
    {
        xGm.SetGlobalBuffer((__gm__ half *)x + BLOCK_LENGTH * AscendC::GetBlockIdx(), BLOCK_LENGTH);
        yGm.SetGlobalBuffer((__gm__ half *)y + BLOCK_LENGTH * AscendC::GetBlockIdx(), BLOCK_LENGTH);
        zGm.SetGlobalBuffer((__gm__ half *)z + BLOCK_LENGTH * AscendC::GetBlockIdx(), BLOCK_LENGTH);
        pipe.InitBuffer(inQueueX, BUFFER_NUM, TILE_LENGTH * sizeof(half));
        pipe.InitBuffer(inQueueY, BUFFER_NUM, TILE_LENGTH * sizeof(half));
        pipe.InitBuffer(outQueueZ, BUFFER_NUM, TILE_LENGTH * sizeof(half));
    }

```

- 算子工程实现需要根据核函数接收的Tiling数据进行动态初始化

```

__aicore__ inline void Init(GM_ADDR x, GM_ADDR y, GM_ADDR z, uint32_t totalLength, uint32_t tileNum)
{
    this->blockLength = totalLength / AscendC::GetBlockNum();
    this->tileNum = tileNum;
    this->tileLength = this->blockLength / tileNum / BUFFER_NUM;

    xGm.SetGlobalBuffer((__gm__ DTYPE_X *)x + this->blockLength * AscendC::GetBlockIdx(), this->blockLength);
    yGm.SetGlobalBuffer((__gm__ DTYPE_Y *)y + this->blockLength * AscendC::GetBlockIdx(), this->blockLength);
    zGm.SetGlobalBuffer((__gm__ DTYPE_Z *)z + this->blockLength * AscendC::GetBlockIdx(), this->blockLength);
    pipe.InitBuffer(inQueueX, BUFFER_NUM, this->tileLength * sizeof(DTYPE_X));
    pipe.InitBuffer(inQueueY, BUFFER_NUM, this->tileLength * sizeof(DTYPE_Y));
    pipe.InitBuffer(outQueueZ, BUFFER_NUM, this->tileLength * sizeof(DTYPE_Z));
}
```


在Process函数内，我们通过Tiling实现设置的tileNum来循环计算每个Tiling块数据。
```
    __aicore__ inline void Process()
    {
        for (int32_t i = 0; i < this->tileNum; i++) {
            CopyIn(i);
            Compute(i);
            CopyOut(i);
        }
    }
```
相对于之前的核函数实现的算子类，算子工程算子类的CopyIn、Compute、CopyOut无变化，这里不做赘述。具体kernel实现代码如下：


In [ ]:
%%writefile Sources/03.02/custom_op/op_kernel/add_custom_template.cpp 
#include "kernel_operator.h"
#include "add_custom_template_tiling.h"
#include "kernel_operator_dump_tensor_intf_impl.h"
constexpr int32_t BUFFER_NUM = 1;  // tensor num for each queue

template <class dtypeX, class dtypeY, class dtypeZ>
class KernelAdd {
public:
    __aicore__ inline KernelAdd() {}
    __aicore__ inline void Init(GM_ADDR x, GM_ADDR y, GM_ADDR z, uint32_t totalLength, uint32_t tileNum)
    {
        this->blockLength = totalLength / AscendC::GetBlockNum();
        this->tileNum = tileNum;
        this->tileLength = this->blockLength / tileNum / BUFFER_NUM;
        xGm.SetGlobalBuffer((__gm__ dtypeX *)x + this->blockLength * AscendC::GetBlockIdx(), this->blockLength);
        yGm.SetGlobalBuffer((__gm__ dtypeY *)y + this->blockLength * AscendC::GetBlockIdx(), this->blockLength);
        zGm.SetGlobalBuffer((__gm__ dtypeZ *)z + this->blockLength * AscendC::GetBlockIdx(), this->blockLength);
        pipe.InitBuffer(inQueueX, BUFFER_NUM, this->tileLength * sizeof(dtypeX));
        pipe.InitBuffer(inQueueY, BUFFER_NUM, this->tileLength * sizeof(dtypeY));
        pipe.InitBuffer(outQueueZ, BUFFER_NUM, this->tileLength * sizeof(dtypeZ));
    }

    __aicore__ inline void Process()
    {
        int32_t loopCount = this->tileNum * BUFFER_NUM;
        for (int32_t i = 0; i < loopCount; i++) {
            CopyIn(i);
            Compute(i);
            CopyOut(i);
        }
        AscendC::printf("Core %d executed %d times in total\n",  AscendC::GetBlockIdx(), loopCount);
    }

private:
    __aicore__ inline void CopyIn(int32_t progress)
    {
        AscendC::LocalTensor<dtypeX> xLocal = inQueueX.AllocTensor<dtypeX>();
        AscendC::LocalTensor<dtypeY> yLocal = inQueueY.AllocTensor<dtypeY>();
        AscendC::DataCopy(xLocal, xGm[progress * this->tileLength], this->tileLength);
        AscendC::DataCopy(yLocal, yGm[progress * this->tileLength], this->tileLength);
        inQueueX.EnQue(xLocal);
        inQueueY.EnQue(yLocal);
    }
    __aicore__ inline void Compute(int32_t progress)
    {
        AscendC::LocalTensor<dtypeX> xLocal = inQueueX.DeQue<dtypeX>();
        AscendC::LocalTensor<dtypeY> yLocal = inQueueY.DeQue<dtypeY>();
        AscendC::LocalTensor<dtypeZ> zLocal = outQueueZ.AllocTensor<dtypeZ>();
        AscendC::Add(zLocal, xLocal, yLocal, this->tileLength);
        outQueueZ.EnQue<dtypeZ>(zLocal);
        inQueueX.FreeTensor(xLocal);
        inQueueY.FreeTensor(yLocal);
    }
    __aicore__ inline void CopyOut(int32_t progress)
    {
        AscendC::LocalTensor<dtypeZ> zLocal = outQueueZ.DeQue<dtypeZ>();
        AscendC::DataCopy(zGm[progress * this->tileLength], zLocal, this->tileLength);
        outQueueZ.FreeTensor(zLocal);
    }

private:
    AscendC::TPipe pipe;
    AscendC::TQue<AscendC::TPosition::VECIN, BUFFER_NUM> inQueueX;
    AscendC::TQue<AscendC::TPosition::VECIN, BUFFER_NUM> inQueueY;
    AscendC::TQue<AscendC::TPosition::VECOUT, BUFFER_NUM> outQueueZ;
    AscendC::GlobalTensor<dtypeX> xGm;
    AscendC::GlobalTensor<dtypeY> yGm;
    AscendC::GlobalTensor<dtypeZ> zGm;
    uint32_t blockLength;
    uint32_t tileNum;
    uint32_t tileLength;
};


 __global__ __aicore__ void add_custom_template(GM_ADDR x, GM_ADDR y, GM_ADDR z, GM_ADDR workspace, GM_ADDR tiling)
{
    REGISTER_TILING_DEFAULT(TilingDataTemplate);
    GET_TILING_DATA_WITH_STRUCT(TilingDataTemplate, tiling_data, tiling);
    KernelAdd<DTYPE_X, DTYPE_Y, DTYPE_Z> op;
    op.Init(x, y, z, tiling_data.totalLength, tiling_data.tileNum);
    op.Process();
}


为验证 AddCustomTemplate 算子工程的可用性，我们将运行预先编写的单算子API调用程序，对该算子工程进行测试，检查其能否正常执行。

首先执行以下命令进行算子包编译。

In [ ]:
#编译算子工程并部署编译出的算子包
!cd Sources/03.02/custom_op;bash build.sh;

然后部署编译出的算子包，算子包有两种部署方式，方式一默认部署，直接执行算子包，例如：
```
./build_out/custom_opp*.run
```
方式二，指定目录部署算子包，命令为:
```
./build_out/custom_opp*.run --install-path=指定的文件夹目录
```
这里我们使用指定目录安装算子包

In [ ]:
#编译算子工程并部署编译出的算子包
!./Sources/03.02/custom_op/build_out/custom_opp*.run --install-path=${HOME}/

接下来运行预先编写的单算子API调用程序，对该算子工程进行测试，检查其能否正常执行。

In [ ]:
# 编译并运行测试代码
!cp -r src/custom_op/test Sources/03.02/test;cd Sources/03.02/test
# 编译测试代码
!g++ -I$ASCEND_TOOLKIT_HOME/include -I${HOME}/vendors/customize/op_api/include -L$ASCEND_TOOLKIT_HOME/lib64 -L${HOME}/vendors/customize/op_api/lib Sources/03.02/test/main.cpp -lcust_opapi -lnnopbase -lacl_rt -o Sources/03.02/execute_add_op;
# 设置自定义算子so路径并执行调用代码
!source ${HOME}/vendors/customize/bin/set_env.bash;./Sources/03.02/execute_add_op

调用程序正常执行后会有如下打印：
```

result is:
3.0 3.0 3.0 3.0 3.0 3.0 3.0 3.0 3.0 3.0
test pass

```

---


## **课后实践**
请根据已提供的SubCustomTemplate算子原型json文件，完成算子工程创建，并完成功能实现。

算子原型json文件：

In [ ]:
%%writefile Sources/03.02/sub_custom_template.json
[{
    "op": "SubCustomTemplate",
    "input_desc": [{
            "name": "x",
            "param_type": "required",
            "format": ["ND", "ND"],
            "type": ["float16", "float"]
        },
        {
            "name": "y",
            "param_type": "required",
            "format": ["ND", "ND"],
            "type": ["float16", "float"]
        }
    ],
    "output_desc": [{
        "name": "z",
        "param_type": "required",
        "format": ["ND", "ND"],
        "type": ["float16", "float"]
    }]
}]

使用msopgen生成算子工程

In [ ]:
# 清除已有的custom_op目录
!rm -rf Sources/03.02/custom_op

# 使用msopgen工具创建算子工程前设置使用开源仓编译的msopgen工具，不设置时默认使用CANN安装目录下的msopgen工具
!export PATH=/usr/local/bin:~/.local/bin:$PATH;msopgen gen -i Sources/03.02/sub_custom_template.json -c ai_core-ascend910b1 -lan cpp -out Sources/03.02/custom_op


修改host侧实现

In [ ]:
%%writefile Sources/03.02/custom_op/op_host/sub_custom_template.cpp

#include "../op_kernel/sub_custom_template_tiling.h"
#include "register/op_def_registry.h"


namespace optiling {
static ge::graphStatus TilingFunc(gert::TilingContext* context)
{

  SubCustomTemplateTilingData *tiling = context->GetTilingData<SubCustomTemplateTilingData>();
  const gert::StorageShape* x1_shape = context->GetInputShape(0);
  int32_t data_sz = 1;
  for (int i = 0; i < x1_shape->GetStorageShape().GetDimNum(); i++)
    data_sz *= x1_shape->GetStorageShape().GetDim(i);
  tiling->size = data_sz;
  context->SetBlockDim(8);
  size_t *currentWorkspace = context->GetWorkspaceSizes(1);
  currentWorkspace[0] = 0;
  return ge::GRAPH_SUCCESS;
}
}


namespace ge {
static ge::graphStatus InferShape(gert::InferShapeContext* context)
{
    const gert::Shape* x1_shape = context->GetInputShape(0);
    gert::Shape* y_shape = context->GetOutputShape(0);
    *y_shape = *x1_shape;
    return GRAPH_SUCCESS;
}
static ge::graphStatus InferDataType(gert::InferDataTypeContext *context)
{
    const auto inputDataType = context->GetInputDataType(0);
    context->SetOutputDataType(0, inputDataType);
    return ge::GRAPH_SUCCESS;
}
}


namespace ops {
class SubCustomTemplate : public OpDef {
public:
    explicit SubCustomTemplate(const char* name) : OpDef(name)
    {
        this->Input("x")
            .ParamType(REQUIRED)
            .DataType({ge::DT_FLOAT16, ge::DT_FLOAT})
            .Format({ge::FORMAT_ND, ge::FORMAT_ND})
            .UnknownShapeFormat({ge::FORMAT_ND, ge::FORMAT_ND});
        this->Input("y")
            .ParamType(REQUIRED)
            .DataType({ge::DT_FLOAT16, ge::DT_FLOAT})
            .Format({ge::FORMAT_ND, ge::FORMAT_ND})
            .UnknownShapeFormat({ge::FORMAT_ND, ge::FORMAT_ND});
        this->Output("z")
            .ParamType(REQUIRED)
            .DataType({ge::DT_FLOAT16, ge::DT_FLOAT})
            .Format({ge::FORMAT_ND, ge::FORMAT_ND})
            .UnknownShapeFormat({ge::FORMAT_ND, ge::FORMAT_ND});

        this->SetInferShape(ge::InferShape).SetInferDataType(ge::InferDataType);

        this->AICore()
            .SetTiling(optiling::TilingFunc);
        this->AICore().AddConfig("ascend910b");

    }
};

OP_ADD(SubCustomTemplate);
}

修改TIling结构体定义

In [ ]:
%%writefile  Sources/03.02/custom_op/op_kernel/sub_custom_template_tiling.h
#ifndef SUB_CUSTOM_TEMPLATE_TILING_H
#define SUB_CUSTOM_TEMPLATE_TILING_H
#include <cstdint>

struct SubCustomTemplateTilingData {
    uint32_t size;
};

#endif

修改kernel实现

In [ ]:
%%writefile  Sources/03.02/custom_op/op_kernel/sub_custom_template.cpp
#include "kernel_operator.h"
#include "sub_custom_template_tiling.h"

extern "C" __global__ __aicore__ void sub_custom_template(GM_ADDR x, GM_ADDR y, GM_ADDR z, GM_ADDR workspace, GM_ADDR tiling) {
    REGISTER_TILING_DEFAULT(SubCustomTemplateTilingData);
    GET_TILING_DATA(tilingData, tiling);
    // TODO: user kernel impl
}

修改完成后，尝试运行已准备好的输入Shape为[8, 2048]，数据类型为half的测试用例，检查算子是否符合预期。

In [ ]:
# 编译部署修改后的算子
!cd Sources/03.02/custom_op;bash build.sh;./build_out/custom_opp*.run --install-path=${HOME}/

In [ ]:
# 清除已有的测试代码
!rm -rf Sources/03.02/test;
# 复制已有的测试代码
!cp -r src/custom_op/test_sub Sources/03.02/test;
# 编译测试代码
!g++ -I$ASCEND_TOOLKIT_HOME/include -I${HOME}/vendors/customize/op_api/include -L$ASCEND_TOOLKIT_HOME/lib64 -L${HOME}/vendors/customize/op_api/lib Sources/03.02/test/main.cpp -lcust_opapi -lnnopbase -lacl_rt -o Sources/03.02/execute_sub_op;
# 设置自定义算子so路径并执行调用代码
!source ${HOME}/vendors/customize/bin/set_env.bash;./Sources/03.02/execute_sub_op

预期窗口输出：
```
result is:
-1.0 -1.0 -1.0 -1.0 -1.0 -1.0 -1.0 -1.0 -1.0 -1.0
test pass

```

执行以下代码可以查看答案

host侧实现

In [ ]:
!cat ./answer/03.02_answer/op_host/sub_custom_template.cpp

TIling结构体定义

In [ ]:
!cat ./answer/03.02_answer/op_kernel/sub_custom_template_tiling.h

Kernel侧实现

In [ ]:
!cat ./answer/03.02_answer/op_kernel/sub_custom_template.cpp